# # Cross-domain transfer: DALES-pretrained -> [Vancouver LiDAR 2022](https://opendata.vancouver.ca/explore/dataset/lidar-2022/map/?location=12,49.25683,-123.14421)

## Reading and visualizing raw point clouds using `Data` objects

### Preparing a `Data` reader

In [ ]:
import os
import sys

# Add the project's files to the python path
# file_path = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))  # for .py script
file_path = os.path.dirname(os.path.abspath(''))  # for .ipynb notebook
sys.path.append(file_path)

import laspy
import torch
from src.data import Data
from src.utils.color import to_float_rgb


def read_vancouver_tile(
        filepath, 
        xyz=True, 
        rgb=True, 
        intensity=True, 
        semantic=True, 
        instance=False,
        remap=True, 
        max_intensity=600):
    """Read a Vancouver tile saved as LAS.

    :param filepath: str
        Absolute path to the LAS file
    :param xyz: bool
        Whether XYZ coordinates should be saved in the output Data.pos
    :param rgb: bool
        Whether RGB colors should be saved in the output Data.rgb
    :param intensity: bool
        Whether intensity should be saved in the output Data.rgb
    :param semantic: bool
        Whether semantic labels should be saved in the output Data.y
    :param instance: bool
        Whether instance labels should be saved in the output Data.obj
    :param remap: bool
        Whether semantic labels should be mapped from their Vancouver ID
        to their train ID
    :param max_intensity: float
        Maximum value used to clip intensity signal before normalizing 
        to [0, 1]
    """
    # Create an emty Data object
    data = Data()
    
    las = laspy.read(filepath)

    # Populate data with point coordinates 
    if xyz:
        # Apply the scale provided by the LAS header
        pos = torch.stack([
            torch.tensor(las[axis])
            for axis in ["X", "Y", "Z"]], dim=-1)
        pos *= las.header.scale
        pos_offset = pos[0]
        data.pos = (pos - pos_offset).float()
        data.pos_offset = pos_offset

    # Populate data with point RGB colors
    if rgb:
        # RGB stored in uint16 lives in [0, 65535]
        data.rgb = to_float_rgb(torch.stack([
            torch.FloatTensor(las[axis].astype('float32') / 65535)
            for axis in ["red", "green", "blue"]], dim=-1))

    # Populate data with point LiDAR intensity
    if intensity:
        # Heuristic to bring the intensity distribution in [0, 1]
        data.intensity = torch.FloatTensor(
            las['intensity'].astype('float32')
        ).clip(min=0, max=max_intensity) / max_intensity

    # Populate data with point semantic segmentation labels
    if semantic:
        y = torch.LongTensor(las['classification'])
        data.y = torch.from_numpy(ID2TRAINID)[y] if remap else y

    # Populate data with point panoptic segmentation labels
    if instance:
        raise NotImplementedError("The dataset does not contain instance labels.")

    return data

In [ ]:
import numpy as np

# Number of classes in the dataset (excluding void/unlabeled/ignored)
VANCOUVER_NUM_CLASSES = 6

# Mapping from original classes
ID2TRAINID = np.asarray([
    VANCOUVER_NUM_CLASSES,  # 0 Not used         ->  6 Ignored
    5,                      # 1 Other            ->  5 Other
    0,                      # 2 Ground           ->  0 Ground
    3,                      # 3 Low vegetation   ->  3 Low vegetation
    VANCOUVER_NUM_CLASSES,  # 4 Unknown / Noise  ->  6 Ignored
    2,                      # 5 High vegetation  ->  2 High vegetation
    4,                      # 6 Building         ->  4 Buildings
    VANCOUVER_NUM_CLASSES,  # 7 Unknown / Noise  ->  6 Ignored
    VANCOUVER_NUM_CLASSES,  # 8 Unknown / Noise  ->  6 Ignored
    1])                     # 9 Water            ->  1 Water

# Class names (including void/unlabeled/ignored last)
VANCOUVER_CLASS_NAMES = [
    'Ground',
    'Water',
    'High vegetation',
    'Low vegetation',
    'Buildings',
    'Other',
    'Ignored']

# Class color palette (including void/unlabeled/ignored last)
VANCOUVER_CLASS_COLORS = np.asarray([
    [243, 214, 171],
    [169, 222, 249],
    [ 70, 115,  66],
    [204, 213, 174],
    [214,  66,  54],
    [186, 160, 164],
    [  0,   0,   0]])

### `Data` visualization

In [ ]:
filepath = '/path/to/your/vancouver.las'
data = read_vancouver_tile(filepath)

In [ ]:
data

In [ ]:
data.num_points

In [ ]:
data.keys

In [ ]:
data.show(class_names=VANCOUVER_CLASS_NAMES, class_colors=VANCOUVER_CLASS_COLORS)

In [ ]:
data.show(center=[425, 282, 15], radius=30, keys=['intensity'], class_names=VANCOUVER_CLASS_NAMES, class_colors=VANCOUVER_CLASS_COLORS)

## Tiling very large point clouds

In [ ]:
from src.transforms import SampleXYTiling, GridSampling3D
from src.data import Batch

# Tile the cloud into `xy_tiling` XY-oriented chunks of equal horizontal 
# span
xy_tiling = (4, 2)

# Voxelize the point cloud only for the sake of faster computation and 
# visualization here
data_5m = GridSampling3D(10)(data)

# Compute each chunk 
chunks = []
for x in range(xy_tiling[0]):
    for y in range(xy_tiling[1]):        
        # Extract the chunk at (x, y) in the tiling grid
        chunk = SampleXYTiling(x=x, y=y, tiling=xy_tiling)(data_5m)

        # Add a 'tile' attribute to the points for visualization
        chunk.tile = torch.full((chunk.num_points,), x * xy_tiling[1] + y)
        
        # Store the chunk for later aggregation
        chunks.append(chunk)

# Aggregate all chunk `Data` objects into one big `Data` object
data_tiled = Batch.from_data_list(chunks)

# Show the resulting `Data' with the 'tile' attribute
data_tiled.show(keys='tile')

In [ ]:
from src.transforms import SampleRecursiveMainXYAxisTiling, GridSampling3D
from src.data import Batch

# Recursively tile the cloud into `2**pc_tiling` chunks with respect to 
# principal components of the XY coordiantes
pc_tiling = 3

# Voxelize the point cloud only for the sake of faster computation and 
# visualization here
data_5m = GridSampling3D(5)(data)

# Compute each chunk 
chunks = []
for x in range(2**pc_tiling):
    # Extract the chunk at x in the recursive tiling
    chunk = SampleRecursiveMainXYAxisTiling(x=x, steps=pc_tiling)(data_5m)

    # Add a 'tile' attribute to the points for visualization
    chunk.tile = torch.full((chunk.num_points,), x)
    
    # Store the chunk for later aggregation
    chunks.append(chunk)

# Aggregate all chunk `Data` objects into one big `Data` object
data_tiled = Batch.from_data_list(chunks)

# Show the resulting `Data' with the 'tile' attribute
data_tiled.show(keys='tile')

In [ ]:
from src.transforms import SampleXYTiling

# Extract the chunk at (x, y) in the tiling grid
data = SampleXYTiling(x=1, y=1, tiling=3)(data)

## Using a pretrained model for inference

### Instantiating transforms from `configs/`

In [ ]:
from src.utils import init_config

cfg = init_config(overrides=[f"experiment=semantic/dales"])

In [ ]:
cfg.keys()

In [ ]:
from src.transforms import instantiate_datamodule_transforms

transforms_dict = instantiate_datamodule_transforms(cfg.datamodule)
transforms_dict

### Applying transforms

In [ ]:
# Apply pre-transforms
nag = transforms_dict['pre_transform'](data)

# Simulate the behavior of the dataset's I/O behavior with only
# `point_load_keys` and `segment_load_keys` loaded from disk
from src.transforms import NAGRemoveKeys
nag = NAGRemoveKeys(level=0, keys=[k for k in nag[0].keys if k not in cfg.datamodule.point_load_keys])(nag)
nag = NAGRemoveKeys(level='1+', keys=[k for k in nag[1].keys if k not in cfg.datamodule.segment_load_keys])(nag)

# Move to device
nag = nag.cuda()

# Apply on-device transforms
nag = transforms_dict['on_device_test_transform'](nag)

In [ ]:
nag

In [ ]:
nag.show(class_names=VANCOUVER_CLASS_NAMES, class_colors=VANCOUVER_CLASS_COLORS, center=[485, 505, 0], radius=20, keys=nag[0].keys, centroids=True, h_edge=True)

### Instantiating a pretrained model from `configs/` and a `*.ckpt`

In [ ]:
import hydra 
from src.utils import init_config

# Path to the checkpoint file downloaded from https://zenodo.org/records/8042712
ckpt_path = "/path/to/your/superpoint_transformer.ckpt"

cfg = init_config(overrides=[f"experiment=semantic/dales"])

# Instantiate the model and load pretrained weights
model = hydra.utils.instantiate(cfg.model)
model = model._load_from_checkpoint(ckpt_path)

### Applying SPT

In [ ]:
# Set the model in inference mode on the same device as the input
model = model.eval().to(nag.device)

# Inference, returns a task-specific ouput object carrying predictions
with torch.no_grad():
    output = model(nag)

In [ ]:
output.semantic_pred().shape, nag.num_points

In [ ]:
# Compute the level-0 (voxel-wise) semantic segmentation predictions 
# based on the predictions on level-1 superpoints and save those for 
# visualization in the level-0 Data under the 'semantic_pred' attribute
nag[0].semantic_pred = output.voxel_semantic_pred(super_index=nag[0].super_index)

In [ ]:
from src.datasets.dales import CLASS_NAMES as DALES_CLASS_NAMES
from src.datasets.dales import CLASS_COLORS as DALES_CLASS_COLORS

nag.show(class_names=DALES_CLASS_NAMES, class_colors=DALES_CLASS_COLORS, center=[485, 505, 0], radius=20)

## Ablations

In [ ]:
import inspect
read_vancouver_tile_source = inspect.getsource(read_vancouver_tile)

print(filepath)
print(ckpt_path)

In [ ]:
import inspect
from src.transforms import *


reader_source = inspect.getsource(read_vancouver_tile)
# i wrote a script to avoid OOM errors  

script_template = f"""
import sys, os, json, time
sys.path.append(os.path.expanduser('~/superpoint_transformer'))

import torch
import numpy as np
import laspy
from copy import deepcopy
from src.data import Data
from src.utils import init_config
from src.utils.color import to_float_rgb
from src.transforms import *

# Reader function
{reader_source}

# Config
VANCOUVER_NUM_CLASSES = 6
ID2TRAINID = np.asarray([VANCOUVER_NUM_CLASSES, 5, 0, 3, VANCOUVER_NUM_CLASSES, 2, 4, VANCOUVER_NUM_CLASSES, VANCOUVER_NUM_CLASSES, 1])
DALES_NUM_CLASSES = 8
VANCOUVER_TO_DALES = np.asarray([0, DALES_NUM_CLASSES, 1, 1, 7, DALES_NUM_CLASSES, DALES_NUM_CLASSES])

# Read args
reg = [float(x) for x in sys.argv[1:4]]
filepath = sys.argv[4]
ckpt_path = sys.argv[5]

# Init transforms
cfg = init_config(overrides=["experiment=semantic/dales"])
transforms_dict = instantiate_datamodule_transforms(cfg.datamodule)

# Read and tile data
data_raw = read_vancouver_tile(filepath)
data_raw = SampleXYTiling(x=1, y=1, tiling=3)(data_raw)

# Override regularization in pre_transform
pre_t = deepcopy(transforms_dict['pre_transform'])
for t in pre_t.transforms:
    if isinstance(t, CutPursuitPartition):
        t.regularization = reg

# Partition
t0 = time.time()
nag_i = pre_t(data_raw)
partition_time = time.time() - t0

num_sp_l1 = nag_i[1].num_points
oracle = nag_i[1].semantic_segmentation_oracle(VANCOUVER_NUM_CLASSES)
oracle_miou = oracle['miou'].item()

# Load model
import hydra
model = hydra.utils.instantiate(cfg.model)
model = model._load_from_checkpoint(ckpt_path)

# Prepare for inference
nag_inf = nag_i.clone()
del nag_i, data_raw
nag_inf = NAGRemoveKeys(level=0, keys=[k for k in nag_inf[0].keys if k not in cfg.datamodule.point_load_keys])(nag_inf)
nag_inf = NAGRemoveKeys(level='1+', keys=[k for k in nag_inf[1].keys if k not in cfg.datamodule.segment_load_keys])(nag_inf)
nag_inf = nag_inf.cuda()
nag_inf = transforms_dict['on_device_test_transform'](nag_inf)

# Inference
model = model.eval().cuda()
with torch.no_grad():
    output = model(nag_inf)

preds = output.voxel_semantic_pred(super_index=nag_inf[0].super_index).cpu().numpy()
gt_hist = nag_inf[0].y.cpu()
vancouver_labels = gt_hist.argmax(dim=1).numpy()
ground_truth = VANCOUVER_TO_DALES[vancouver_labels]

mask = ground_truth < DALES_NUM_CLASSES
gt_valid = ground_truth[mask]
pred_valid = preds[mask]

ious = {{}}
for cls_id, cls_name in [(0, "Ground"), (1, "Vegetation"), (7, "Buildings")]:
    inter = ((pred_valid == cls_id) & (gt_valid == cls_id)).sum()
    union = ((pred_valid == cls_id) | (gt_valid == cls_id)).sum()
    ious[cls_name] = float(inter / union) if union > 0 else 0.0

actual_miou = float(np.mean(list(ious.values())))

# Output as JSON on last line
print(json.dumps({{
    "regularization": reg[0],
    "num_sp_l1": int(num_sp_l1),
    "oracle_miou": float(oracle_miou),
    "actual_miou": actual_miou,
    "partition_time": float(partition_time),
    "ious": ious
}}))
"""

with open(os.path.expanduser('~/superpoint_transformer/run_partition.py'), 'w') as f:
    f.write(script_template)

print("Script written successfully")

In [ ]:
import subprocess
import json
import os

filepath = '/Data/aziz.bacha/datasets/495000_5457000/495000_5457000.las'
ckpt_path = "/users/eleves-b/2022/aziz.bacha/superpoint_transformer/models/spt-2_dales.ckpt"

regularizations = [
    [0.05, 0.1, 0.15],
    [0.1, 0.2, 0.3],
    [0.15, 0.3, 0.45],
    [0.2, 0.4, 0.6],
    [0.3, 0.6, 0.9],
    [0.5, 1.0, 1.5],
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
]

results = []

for i, reg in enumerate(regularizations):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(regularizations)}] Regularization: {reg}")
    print(f"{'='*60}")

    cmd = [
        'python', os.path.expanduser('~/superpoint_transformer/run_partition.py'),
        str(reg[0]), str(reg[1]), str(reg[2]),
        filepath,
        ckpt_path
    ]

    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

    if result.returncode == 0:
        lines = result.stdout.strip().split('\n')
        r = json.loads(lines[-1])
        results.append(r)

        print(f"  Superpoints L1:  {r['num_sp_l1']:,}")
        print(f"  Partition time:  {r['partition_time']:.1f}s")
        print(f"  Oracle mIoU:     {r['oracle_miou']:.2f}%")
        print(f"  Actual mIoU:     {r['actual_miou']*100:.2f}%")
        print(f"  Per-class IoU:")
        for cls_name, iou in r['ious'].items():
            print(f"    {cls_name:<12} {iou*100:.2f}%")
    else:
        print(f"  FAILED:\n{result.stderr[-1000:]}")
        results.append({
            'regularization': reg[0],
            'num_sp_l1': 0,
            'oracle_miou': 0,
            'actual_miou': None,
            'partition_time': 0,
            'ious': {}
        })

# Summary table
print(f"\n{'='*130}")
print(f"  PARTITION SENSITIVITY STUDY - DALES model on Vancouver")
print(f"{'='*130}")
print(f"{'Reg':>6} | {'#SP L1':>10} | {'Oracle mIoU':>12} | {'Actual mIoU':>12} | {'Ground':>9} | {'Vegetation':>11} | {'Buildings':>10} | {'Time':>6}")
print(f"{'-'*130}")
for r in results:
    if r['actual_miou'] is not None:
        actual = f"{r['actual_miou']*100:>11.2f}%"
        ground = f"{r['ious'].get('Ground', 0)*100:>8.2f}%"
        veg = f"{r['ious'].get('Vegetation', 0)*100:>10.2f}%"
        build = f"{r['ious'].get('Buildings', 0)*100:>9.2f}%"
    else:
        actual = f"{'FAILED':>12}"
        ground = f"{'-':>9}"
        veg = f"{'-':>11}"
        build = f"{'-':>10}"
    print(f"{r['regularization']:>6.2f} | {r['num_sp_l1']:>10,} | {r['oracle_miou']:>11.2f}% | {actual} | {ground} | {veg} | {build} | {r['partition_time']:>5.1f}s")

# Best result
valid = [r for r in results if r['actual_miou'] is not None]
if valid:
    best = max(valid, key=lambda r: r['actual_miou'])
    print(f"\n  Best actual mIoU: {best['actual_miou']*100:.2f}% at regularization={best['regularization']:.2f} ({best['num_sp_l1']:,} superpoints)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

valid = [r for r in results if r['actual_miou'] is not None]
regs = [r['regularization'] for r in valid]
oracle = [r['oracle_miou'] for r in valid]
actual = [r['actual_miou'] * 100 for r in valid]
num_sp = [r['num_sp_l1'] for r in valid]
ground = [r['ious']['Ground'] * 100 for r in valid]
veg = [r['ious']['Vegetation'] * 100 for r in valid]
build = [r['ious']['Buildings'] * 100 for r in valid]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Oracle vs Actual mIoU
ax = axes[0]
ax.plot(regs, oracle, 'b-o', label='Oracle mIoU', linewidth=2)
ax.plot(regs, actual, 'r-o', label='Actual mIoU', linewidth=2)
ax.fill_between(regs, actual, oracle, alpha=0.15, color='gray', label='Gap')
ax.set_xlabel('Regularization (level 1)')
ax.set_ylabel('mIoU (%)')
ax.set_title('Oracle vs Actual mIoU')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Per-class IoU
ax = axes[1]
ax.plot(regs, ground, 's-', label='Ground', linewidth=2, color='#F3D6AB')
ax.plot(regs, veg, '^-', label='Vegetation', linewidth=2, color='#467342')
ax.plot(regs, build, 'D-', label='Buildings', linewidth=2, color='#D64236')
ax.set_xlabel('Regularization (level 1)')
ax.set_ylabel('IoU (%)')
ax.set_title('Per-class IoU')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Superpoints vs mIoU
ax = axes[2]
ax.plot(num_sp, oracle, 'b-o', label='Oracle mIoU', linewidth=2)
ax.plot(num_sp, actual, 'r-o', label='Actual mIoU', linewidth=2)
ax.set_xlabel('Number of superpoints (L1)')
ax.set_ylabel('mIoU (%)')
ax.set_title('Partition Size vs Quality')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k'))

plt.tight_layout()
plt.savefig('partition_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from copy import deepcopy
import time
from src.transforms import *


feature_configs = {
    'Default (lin+plan+scat+elev)': ['linearity', 'planarity', 'scattering', 'elevation'],
    'No elevation (lin+plan+scat)': ['linearity', 'planarity', 'scattering'],
    'No geometry (elev only)':      ['elevation'],
    'Geometry only (lin+plan+scat)': ['linearity', 'planarity', 'scattering'],
    'All + color':                   ['linearity', 'planarity', 'scattering', 'elevation', 'rgb'],
    'Color only (rgb)':              ['rgb'],
    'Minimal (plan+elev)':           ['planarity', 'elevation'],
}

results_feat = []

for name, features in feature_configs.items():
    print(f"\n{'='*60}")
    print(f"Features: {name}")
    print(f"{'='*60}")

    data_raw = read_vancouver_tile(filepath)
    data_raw = SampleXYTiling(x=1, y=1, tiling=3)(data_raw)

    # Voxelization
    data_i = GridSampling3D(size=0.1, hist_key='y', hist_size=VANCOUVER_NUM_CLASSES + 1)(data_raw)
    # Neighbors
    data_i = KNN(k=25, r_max=2)(data_i)
    # Elevation
    data_i = GroundElevation(threshold=5, scale=20)(data_i)
    # Point features
    data_i = PointFeatures(keys=('elevation', 'rgb', 'hsv', 'linearity', 'planarity', 'scattering', 'verticality'))(data_i)
    # Adjacency graph
    data_i = AdjacencyGraph(k=10, w=1)(data_i)
    # Copy selected features to x
    data_i = AddKeysTo(keys=features, to='x', delete_after=False)(data_i)

    # Partition with default regularization
    t0 = time.time()
    nag_i = CutPursuitPartition(
        regularization=[0.1, 0.2, 0.3],
        spatial_weight=[0.1, 0.01, 0.001],
        cutoff=[10, 30, 100],
        iterations=15,
        k_adjacency=10)(data_i)
    partition_time = time.time() - t0

    num_sp_l1 = nag_i[1].num_points
    ratios = nag_i.level_ratios
    oracle = nag_i[1].semantic_segmentation_oracle(VANCOUVER_NUM_CLASSES)
    oracle_miou = oracle['miou'].item()

    print(f"  Superpoints L1: {num_sp_l1:,}")
    print(f"  Level ratios:   {ratios}")
    print(f"  Oracle mIoU:    {oracle_miou:.2f}%")
    print(f"  Time:           {partition_time:.1f}s")

    results_feat.append({
        'name': name,
        'features': features,
        'num_sp_l1': num_sp_l1,
        'oracle_miou': oracle_miou,
        'partition_time': partition_time,
    })

    del nag_i, data_i, data_raw

# Summary
print(f"\n{'='*100}")
print(f"  FEATURE ABLATION STUDY — Oracle mIoU on Vancouver")
print(f"{'='*100}")
print(f"{'Config':<35} | {'#SP L1':>10} | {'Oracle mIoU':>12} | {'Time':>6}")
print(f"{'-'*100}")
for r in results_feat:
    print(f"{r['name']:<35} | {r['num_sp_l1']:>10,} | {r['oracle_miou']:>11.2f}% | {r['partition_time']:>5.1f}s")

best = max(results_feat, key=lambda r: r['oracle_miou'])
worst = min(results_feat, key=lambda r: r['oracle_miou'])
print(f"\n  Best:  {best['name']} ({best['oracle_miou']:.2f}%)")
print(f"  Worst: {worst['name']} ({worst['oracle_miou']:.2f}%)")